# RHO-FFT · P2（Colab）：剂量响应实验（E2），检验 H2

这个 notebook 在一次会话里先确认基线没有被改坏（P0 的回归检查），再跑剂量响应实验：稀缺层上有一个不可靠的一阶来源时，损害怎样随"同层还有几份正常的一阶报告"变化，不同聚合规则各自如何。全部逻辑在 `experiments/p2/p2.py` 和 `ho_fl/dose.py` 里。

**运行前**

1. 运行时类型选 GPU（默认 G4）。
2. 代码只从 GitHub 读取（分支见参数 `REPO_REF`）。仓库私有时在 Colab 的"密钥"里添加 `GITHUB_TOKEN`。
3. 不使用 Google Drive。结果留在本页面的输出里，每个阶段结束时浏览器会下载一个结果压缩包。会话之间不保存任何东西。
4. 从上到下运行全部单元格。

**实验怎么做**

- 先跑一次 HO-FL 默认设置的 40 轮，把第 40 轮的全局状态作为所有实验臂共同的起点。
- 每个实验臂从这个起点再训练 20 轮。每轮固定选中一台指定的强设备（记为 f），另外 5 台从其余 29 台里按种子抽取，所有实验臂用同一份名单。
- 剂量 m：这 5 台里有 m 台被设成"在稀缺段 S1 上做一阶更新"，其余在 S1 上是零阶。所以 S1 上的一阶报告共 m+1 份，其中一份来自 f。
- 故障：f 在 S1 上的更新被取反（`flip1`）或取反后放大 20 倍（`flip20`）；每个臂都有一个不加故障的配对运行。
- 聚合规则：

| 名称 | 做法 |
|---|---|
| `mean_all` | HO-FL 原方法：全部报告等权平均 |
| `fo_mean` | 只在一阶报告之间平均 |
| `robust_all` | 全部报告做几何中位数（不区分阶数的稳健聚合） |
| `fo_robust` | 只在一阶报告之间做几何中位数 |
| `quorum_robust` | 一阶报告达到 3 份时用 `fo_robust`，否则退回 `robust_all`。按构造与这两者之一完全相同，不单独运行，表里由它们填入 |
| `quorum_hold` | 关掉零阶更新；一阶报告达到 3 份时做几何中位数，否则这一层本轮不更新 |
| `quorum_dilute` | 关掉零阶更新；不足 3 份时把一阶报告求和后除以 K |

后两种是根据第一次实验加的：零阶报告在这个预算下是纯噪声，"退回全部报告"等于退回噪声。

**读法**

- "相对影响"：故障报告把聚合结果推偏了多少，以它对等权平均的影响为 1。小于 1 说明规则压住了它，大于 1 说明规则放大了它。
- "损害"：有故障时的测试损失减去配对运行的测试损失。每个臂只训练 20 轮，差值本身会比较小。
- H2 预期：`fo_robust` 在 m≥2（一阶报告里正常的占多数）时好于 `robust_all`，在 m≤1 时差于它。

**时间预算**

`BUDGET_HOURS` 是整个 notebook 的总预算。队列按 m = 2、1、3、0、4、5 的顺序排（预期的交叉点在前），放不进预算的运行会被跳过并在报告里列出。

In [ ]:
#@title Parameters { display-mode: "form" }
# The code is read from GitHub only.
REPO_URL = "https://github.com/GuangLun2000/RHO-FFT.git"  #@param {type:"string"}
REPO_REF = "main"  #@param {type:"string"}
ASK_FOR_TOKEN = True  #@param {type:"boolean"}

# Total wall-clock budget of this notebook, in hours.
BUDGET_HOURS = 5.5  #@param {type:"number"}

# "venv": Python 3.11 with the pinned dependencies of pyproject.toml (recommended).
# "system": Colab's own Python and torch; only a fallback, versions then differ from the pins.
ENV_MODE = "venv"  #@param ["venv", "system"]
# PyTorch installed only if the pinned 2.6.0 (CUDA 12.4) cannot use the GPU. The G4 runtime is an RTX PRO 6000 of
# the Blackwell generation, which needs a CUDA 12.8 build (PyTorch 2.7 or later).
TORCH_FALLBACK = "2.7.1"  #@param {type:"string"}

# What to run.
MODEL = "qwen"  #@param ["qwen", "opt"]
SEED = 42  #@param {type:"integer"}
# Doses in queue order (honest first-order reports on the scarce segment next to the unreliable one) and faults.
DOSES = "2,1,3,0,4,5"  #@param {type:"string"}
FAULTS = "none,flip20,flip1"  #@param {type:"string"}
# Runs that may share the GPU (each run is its own process; lowered automatically if memory does not allow it).
JOBS = 2  #@param {type:"integer"}

# Stages.
RUN_TINY_DRY_RUN = True  #@param {type:"boolean"}
RUN_TESTS = True  #@param {type:"boolean"}
RUN_REGRESSION = True  #@param {type:"boolean"}
RUN_DOSE = True  #@param {type:"boolean"}
# Run the experiment even when the regression check did not pass (not recommended).
FORCE = False  #@param {type:"boolean"}
# Download the result bundle through the browser after every stage and before the runtime is released.
DOWNLOAD_BUNDLE = True  #@param {type:"boolean"}
# Release the GPU runtime after the last cell, and after an error in any cell.
RELEASE_GPU = True  #@param {type:"boolean"}

## A1. 会话与 GPU

记录开始时间（时间预算从这里算起），显示 GPU，注册"出错后先保存结果再释放 GPU"的处理器。

In [ ]:
#@title A1 · Session and GPU
import base64
import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

SESSION_START = time.time()
DEADLINE = SESSION_START + 3600 * BUDGET_HOURS
REPO_DIR = "/content/RHO-FFT"
WORK = "/content/work/p2"            # results of the P2 pipeline (local disk, fast)
P0_WORK = "/content/work/p0"         # results of the baseline regression (golden values, verdicts)
TINY_WORK = "/content/work/p2-tiny"  # results of the CPU dry run, kept apart from the real ones
TINY_P0_WORK = "/content/work/p0-tiny"
CACHE = "/content/cache"             # models, datasets, token caches and manifests
VENV = "/content/venv"


def sh(command, cwd=None, env=None, check=True):
    # Run a command, stream its output into the cell, and stop on failure when check is set.
    shown = command if isinstance(command, str) else " ".join(str(part) for part in command)
    print(f"$ {shown}", flush=True)
    process = subprocess.Popen(command, shell=isinstance(command, str), cwd=cwd, env=env, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="", flush=True)
    code = process.wait()
    if check and code != 0:
        raise RuntimeError(f"Command failed with exit code {code}: {shown}")
    return code


def hours_left():
    return (DEADLINE - time.time()) / 3600


STATUS = {}  # one line per stage, printed in the last cell


def release_gpu(delay=10):
    # Hand the GPU runtime back. Everything on the VM's local disk is lost, so this runs only after the results
    # have been shown and downloaded.
    if not RELEASE_GPU:
        print("RELEASE_GPU is off; the runtime stays connected.")
        return
    print(f"Releasing the GPU runtime in {delay} seconds ...", flush=True)
    time.sleep(delay)
    from google.colab import runtime
    runtime.unassign()


def stop(message):
    # A failed check stops "Run all"; the error handler below then releases the GPU.
    raise RuntimeError(message)


def download_results(refresh=True):
    # Send the result bundle to the browser. With refresh, the feedback is rebuilt first from whatever exists, so
    # that an interrupted session still leaves with its finished runs. Returns whether a download was started.
    if refresh and "p2" in globals() and Path(WORK).exists():
        try:
            p2("feedback", extra=["--quiet"])
        except Exception as error:
            print("feedback could not be rebuilt:", repr(error))
    bundle = Path(WORK) / "feedback" / "feedback_P2_bundle.zip"
    if not (DOWNLOAD_BUNDLE and bundle.exists()):
        return False
    try:
        from google.colab import files
        files.download(str(bundle))
        return True
    except Exception as error:
        print("download failed:", repr(error))
        return False


def _release_on_error(shell, etype, value, traceback, tb_offset=None):
    # Any error in any later cell: show it, save what exists, then release the GPU instead of leaving it idle.
    shell.showtraceback((etype, value, traceback), tb_offset=tb_offset)
    download_results()
    release_gpu(30 if DOWNLOAD_BUNDLE else 10)


# Manual interruption (KeyboardInterrupt) is not an Exception and does not release the runtime.
get_ipython().set_custom_exc((Exception,), _release_on_error)


sh("nvidia-smi", check=False)

## A2. 取代码

克隆仓库并检出 `REPO_REF`。回归检查要用 `git archive` 取出两个参照提交（上游原样导入、插桩提交），所以需要带完整历史的克隆。token 只通过环境变量传给 git，不会出现在输出里。

In [ ]:
#@title A2 · Get the code
def git_environment():
    # GitHub credentials for a private repository, passed to git through the environment only.
    env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
    token = None
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    except Exception as error:
        print("Colab secret GITHUB_TOKEN not available:", type(error).__name__)
    if not token and ASK_FOR_TOKEN:
        from getpass import getpass
        token = getpass("GitHub token with read access (leave empty for a public repository): ").strip()
    if token:
        basic = base64.b64encode(f"x-access-token:{token}".encode()).decode()
        env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.https://github.com/.extraheader",
                   GIT_CONFIG_VALUE_0=f"AUTHORIZATION: basic {basic}")
    return env


source, git_env = REPO_URL, git_environment()

if not os.path.isdir(f"{REPO_DIR}/.git"):
    sh(["git", "clone", "--quiet", source, REPO_DIR], env=git_env)
else:
    sh(["git", "-C", REPO_DIR, "fetch", "--quiet", "origin"], env=git_env)
remote_ref = f"origin/{REPO_REF}"
has_remote_ref = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--verify", "--quiet", remote_ref],
                                capture_output=True).returncode == 0
sh(["git", "-C", REPO_DIR, "checkout", "--quiet", "--detach", remote_ref if has_remote_ref else REPO_REF])
sh(["git", "-C", REPO_DIR, "log", "--oneline", "-8"])

revisions = json.loads(Path(f"{REPO_DIR}/experiments/p0/revisions.json").read_text())
for name, sha in revisions.items():
    if subprocess.run(["git", "-C", REPO_DIR, "cat-file", "-e", sha + "^{commit}"], capture_output=True).returncode != 0:
        raise RuntimeError(f"Reference revision {name} ({sha}) is missing from this clone; a full-history clone is required.")
print("reference revisions present:", revisions)

## A3. 环境

建一个独立的 Python 3.11 虚拟环境，按 `pyproject.toml` 的固定版本安装依赖（PyTorch 2.6.0 / CUDA 12.4）。所有训练都在这个环境的子进程里运行。仓库本身不安装，运行时通过 `PYTHONPATH` 指向源码树。

G4 运行时是 Blackwell 架构的 RTX PRO 6000，固定版本的 PyTorch 2.6.0（CUDA 12.4）不支持它。环境建好后会先探测 GPU：固定版本用不了时，自动换成参数 `TORCH_FALLBACK` 指定的版本（CUDA 12.8 构建），并在结果里记录实际使用的版本。回归检查在同一环境里进行，所以基线是否被改坏的结论不受影响；与论文数值的对照则多了一层环境差异。H100 上直接使用固定版本。

In [ ]:
#@title A3 · Environment
def clean_environment():
    # Colab exports constraint files for its own package set (PIP_CONSTRAINT, UV_CONSTRAINT, ...). They must not
    # apply to the separate environment built here, which pins different versions.
    return {key: value for key, value in os.environ.items() if not key.startswith(("UV_", "PIP_"))}


if ENV_MODE == "venv":
    uv = f"{sys.executable} -m uv"
    sh(f"{sys.executable} -m pip install -q uv")
    if not os.path.exists(f"{VENV}/bin/python"):
        if sh(f"{uv} venv --python 3.11 {VENV}", env=clean_environment(), check=False) != 0:
            print("Python 3.11 could not be provisioned; building the environment on the runtime's own Python instead")
            shutil.rmtree(VENV, ignore_errors=True)
            sh(f"{uv} venv --python {sys.executable} {VENV}", env=clean_environment())
    install = f"{uv} pip install --python {VENV}/bin/python -r pyproject.toml --extra dev"
    if sh(install + " --torch-backend cu124", cwd=REPO_DIR, env=clean_environment(), check=False) != 0:
        print("retrying without --torch-backend (the default Linux wheel of torch 2.6.0 is the CUDA 12.4 build)")
        sh(install, cwd=REPO_DIR, env=clean_environment())
    PY = f"{VENV}/bin/python"
else:
    PY = sys.executable
    sh(f"{PY} -m pip install -q transformers==4.56.2 peft==0.17.1 datasets==3.6.0 accelerate==1.10.1 "
       "safetensors==0.6.2 sentencepiece==0.2.1 pytest", env=clean_environment())

sh([PY, "-c", "import torch, transformers, peft, datasets; "
              "print('torch', torch.__version__, '| cuda runtime', torch.version.cuda, '| cuda available', torch.cuda.is_available()); "
              "print('transformers', transformers.__version__, '| peft', peft.__version__, '| datasets', datasets.__version__)"])

# Probe the GPU with the interpreter that will train. The pinned PyTorch has no kernels for Blackwell GPUs (the G4
# runtime); if it cannot use this GPU, the fallback build for CUDA 12.8 is installed and probed again. The versions
# actually used are recorded with every result, and the regression check runs in the same environment.
GPU_PROBE = ("import torch; x = torch.randn(256, 256, device='cuda', dtype=torch.bfloat16); "
             "assert torch.isfinite((x @ x).float()).all(); "
             "print('GPU usable:', torch.cuda.get_device_name(0), '| torch', torch.__version__)")


def gpu_usable():
    return sh([PY, "-c", GPU_PROBE], check=False) == 0


if shutil.which("nvidia-smi") is None:
    stop("No GPU in this runtime. Choose a GPU runtime type and run again.")
if not gpu_usable():
    if ENV_MODE != "venv":
        stop("The runtime's own PyTorch cannot use this GPU. Send the output above back.")
    print(f"\nThe pinned PyTorch cannot use this GPU; installing torch {TORCH_FALLBACK} built for CUDA 12.8 instead.")
    fallback = f"{uv} pip install --python {VENV}/bin/python torch=={TORCH_FALLBACK}"
    if sh(fallback + " --torch-backend cu128", env=clean_environment(), check=False) != 0:
        sh(fallback + " --index-url https://download.pytorch.org/whl/cu128", env=clean_environment())
    if not gpu_usable():
        stop("PyTorch cannot use this GPU with either build. Send the output above back.")
    STATUS["environment note"] = f"torch {TORCH_FALLBACK} (CUDA 12.8) in place of the pinned 2.6.0, which does not support this GPU"


def p0(*stages, root=P0_WORK, extra=()):
    # Run stages of experiments/p0/p0.py (baseline regression) with the training interpreter; 0 = all stages ok.
    command = [PY, "experiments/p0/p0.py", *stages, "--root", root, "--cache-dir", CACHE, "--device", "cuda", *extra]
    return sh(command, cwd=REPO_DIR, check=False)


def p2(*stages, root=WORK, extra=()):
    # Run stages of experiments/p2/p2.py (base run, dose-response arms, feedback); 0 = all stages ok.
    command = [PY, "experiments/p2/p2.py", *stages, "--root", root, "--cache-dir", CACHE, "--device", "cuda",
               "--model", MODEL, "--seed", str(SEED), "--doses", DOSES, "--faults", FAULTS, *extra]
    return sh(command, cwd=REPO_DIR, check=False)


def show_feedback(download=True):
    # Rebuild the feedback from the runs finished so far, show it, and send the bundle to the browser.
    from IPython.display import Markdown, display
    p2("feedback", extra=["--quiet"])
    report = Path(WORK) / "feedback" / "feedback_P2.md"
    if report.exists():
        display(Markdown(report.read_text()))
    if download:
        started = download_results(refresh=False)
        print("result bundle:", "download started" if started else "not downloaded")


def run_stage(stage, label, show=True):
    # One stage of the experiment under the remaining budget, then the feedback so far. Returns whether it finished.
    if regression_ok is False and not FORCE:
        print(f"{label} skipped: the regression check did not pass. Set FORCE = True to run it anyway.")
        STATUS[label] = "skipped (regression did not pass)"
        return False
    if hours_left() < 0.3:
        print(f"{label} skipped: less than 18 minutes of budget left.")
        STATUS[label] = "skipped (no budget left)"
        return False
    if regression_ok is None:
        print("NOTE: the regression check was not run in this session; the results build on an unverified baseline.")
    # Ten minutes of the budget are kept for the feedback and the download.
    done = p2(stage, extra=["--jobs", str(JOBS), "--deadline", str(DEADLINE - 600)]) == 0
    STATUS[label] = "all runs finished" if done else "partly done (see the queue table in the report)"
    print(f"{hours_left():.2f} h of budget left")
    if show:
        show_feedback()
    return done


## B. 预演（CPU，约三分钟）

用随机初始化的小模型和合成数据，把 P0 的回归检查和剂量响应实验的整条流程（起点运行、名单与故障文件、24 个实验臂、反馈）各走一遍。它只检查流程能不能跑通，数字没有意义。失败时 notebook 停下并释放 GPU，请把输出带回。

In [ ]:
#@title B · Dry run on CPU
if RUN_TINY_DRY_RUN:
    code = sh([PY, "experiments/p0/p0.py", "env", "golden", "regress", "--tiny", "--root", TINY_P0_WORK], cwd=REPO_DIR, check=False)
    STATUS["dry run: baseline regression"] = "ok" if code == 0 else "FAILED"
    if code != 0:
        stop("The dry run of the baseline regression failed: on CPU the current code no longer matches the upstream revision. Send the output above back.")
    code = sh([PY, "experiments/p2/p2.py", "all", "--tiny", "--quiet", "--jobs", "2", "--root", TINY_WORK], cwd=REPO_DIR, check=False)
    STATUS["dry run: P2 pipeline"] = "ok" if code == 0 else "FAILED"
    if code != 0:
        stop("The dry run of the P2 pipeline failed. Nothing has been run on the GPU yet; send the output above back.")
    print(f"\ndry run ok, {hours_left():.2f} h of budget left")


## C1. 环境记录与单元测试

记录 GPU 型号、驱动和各依赖的版本，并做一次 BF16 探测；在固定版本的环境里跑一遍不需要 GPU 的测试（约一分钟）。

In [ ]:
#@title C1 · Environment record and unit tests
env_ok = p0("env") == 0
STATUS["environment"] = "ok" if env_ok else "FAILED"
if not env_ok:
    stop("Environment check failed (no CUDA device, bfloat16 not usable, or a reference revision is missing). See the warnings above.")
if RUN_TESTS:
    tests_ok = p0("tests") == 0
    STATUS["unit tests"] = "ok" if tests_ok else "FAILURES"
    print("unit tests:", "ok" if tests_ok else "FAILURES (listed above)")

## C2. 黄金值与基线回归（I1、I2）

三个冒烟配置（HO-FL、FedAvg、DeComFL，OPT-125M / SST-2，3 轮）在上游原样提交上各跑一次，在插桩提交上各跑两次，在当前代码上各跑一次，再加两次 `ho-fl-replay`。当前代码改动了 `hybrid_step` 和 `local_step`（新增默认关闭的关键字参数），这一步检查默认配置下的结果仍与上游一致。

读法：`I1 verdict` 为 `PASS_BITWISE` 或 `PASS_TOLERANCE` 即通过；`FAIL` 说明当前代码改变了基线，后面的实验组默认不再运行；`INCONCLUSIVE` 说明这块 GPU 上参照提交自己两次运行的差就超过了容差，需要研究负责人决定怎么判。

In [ ]:
#@title C2 · Golden values and regression (I1, I2)
regression_ok = None
if RUN_REGRESSION:
    golden_ok = p0("golden") == 0
    regression_ok = golden_ok and p0("regress") == 0
    STATUS["golden values"] = "ok" if golden_ok else "ATTENTION"
    STATUS["regression (I1, I2)"] = "ok" if regression_ok else "ATTENTION"
    p0("feedback", extra=["--quiet"])
    # The regression record travels with the P2 bundle.
    target = Path(WORK) / "p0_feedback"
    target.mkdir(parents=True, exist_ok=True)
    for path in [*Path(P0_WORK).glob("feedback/feedback_P0.*"), *Path(P0_WORK).glob("golden/*/golden.json"),
                 *Path(P0_WORK).glob("regress/*/verdicts.json"), Path(P0_WORK) / "env.json"]:
        if path.exists():
            shutil.copy2(path, target / (path.parent.name + "__" + path.name if path.name in ("golden.json", "verdicts.json") else path.name))
    print("\ngolden:", "ok" if golden_ok else "ATTENTION", "| regression (I1, I2):", "ok" if regression_ok else "ATTENTION")
    print(f"{hours_left():.2f} h of budget left")

## C3. 起点运行（HO-FL 默认设置，40 轮）

它的第 40 轮状态是所有实验臂的共同起点，同时确定三档设备的划分和那台指定的强设备。

In [ ]:
#@title C3 · Base run (the common starting state)
base_ok = run_stage("base", "base run", show=False) if RUN_DOSE else False

## C4. 剂量响应实验臂

每个 m 有 6 种聚合规则乘以 3 种故障条件共 18 个 20 轮的运行。先对三种成本类型各跑 3 轮预检计时，再按 m = 2、1、3、0、4、5 排队。发散的运行记为 `DIVERGED`，它是一种结果，按最差处理。结束后显示目前为止的完整反馈并下载结果包。

In [ ]:
#@title C4 · Dose-response arms
if RUN_DOSE and base_ok:
    run_stage("dose", "dose-response arms")
elif RUN_DOSE:
    print("dose-response arms skipped: the base run did not finish.")
    STATUS["dose-response arms"] = "skipped (no base run)"

## D. 总反馈与剂量响应图

每种故障强度各有四张表（有故障时的测试损失、损害、准确率、在稀缺段上的相对影响），加上不加故障时各规则的测试损失、H2 的判定、队列与耗时、结论。图里左列是相对影响，右列是损害，横轴是 m；竖虚线是法定数的位置。

In [ ]:
#@title D · Feedback report and dose-response figure
from IPython.display import Image, display

show_feedback(download=False)
STATUS["feedback"] = "written"
feedback_dir = Path(WORK) / "feedback"
data = json.loads((feedback_dir / "feedback_P2.json").read_text())
# The figure is drawn by the notebook's own Python, which has matplotlib.
sh([sys.executable, f"{REPO_DIR}/experiments/p2/plot_dose.py", str(feedback_dir / "feedback_P2.json"), str(feedback_dir)], check=False)
if (feedback_dir / "p2_dose.png").exists():
    display(Image(filename=str(feedback_dir / "p2_dose.png")))
print("report:", feedback_dir / "feedback_P2.md")
print(f"session time used: {(time.time() - SESSION_START) / 3600:.2f} h")

## E. 汇总并释放 GPU

打印各步骤的状态，让浏览器下载结果压缩包，然后释放 GPU 运行时。

In [ ]:
#@title E · Summary, download, then release the GPU
print("stage status")
for name, state in STATUS.items():
    print(f"  {name:30s} {state}")
print("H2:", ", ".join(f"{fault} {entry['verdict']}" for fault, entry in data["h2"].items()) or "not computed",
      f"| {data['finished_runs']} of {data['planned_runs']} arms finished")
print(f"session time used: {(time.time() - SESSION_START) / 3600:.2f} h of {BUDGET_HOURS} h")
started = download_results(refresh=True)
print("result bundle:", "download started" if started else "NOT downloaded; the report above is the record of this session")
release_gpu(30 if started else 10)

## 带回代码库的内容

- 页面上显示的 **`feedback_P2.md`**：第 3 节是各故障强度下的四张表，第 5 节是 H2 的判定，第 7 节是结论。
- 浏览器下载的 **`feedback_P2_bundle.zip`**：报告、`dose.csv`、每个运行的 `config.json`、`summary.json`、`rounds.jsonl`、`curves.jsonl`、`ledger.jsonl`、`influence.jsonl`、名单和故障文件、计时和日志，以及 `p0_feedback/` 下的回归记录。
- G2 是说明规定的停点：带回这份结果后再决定端到端韧性实验（E3）的配置。